# Milestone 1 results

Displays what `grapheme_extraction/evaluate.py` wrote to `reports/`. Run that first. **All numbers are on synthetic seals.**

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

R = Path('../reports')
res = json.loads((R / 'eval_results.json').read_text())

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for f in sorted(R.glob('*_results.csv')):
    df = pd.read_csv(f); df.columns = df.columns.str.strip()
    name = f.name.replace('_results.csv', '')
    for ax, col in zip(axes, ['train/box_loss', 'val/box_loss', 'metrics/mAP50(B)']):
        col = col if col in df else col.replace('box_loss', 'giou_loss')  # RT-DETR names its box loss differently
        ax.plot(df['epoch'], df[col], label=name); ax.set_title(col); ax.set_xlabel('epoch')
axes[0].legend(); plt.tight_layout()

## Detector comparison on the held-out synthetic test split (point estimate, 95% bootstrap interval)

In [ ]:
pd.DataFrame({d: {m: f'{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]' for m, v in e['metrics'].items()} | {k: f'{v:.3f}' for k, v in e.get('ap', {}).items()}
              for d, e in res['detectors'].items()})

## Classifier by sign frequency

In [ ]:
pd.DataFrame({k: {'crops': v['crops'], 'top1': round(v['top1'][0], 3), 'top5': round(v['top5'][0], 3)} for k, v in res['classifier'].items()}).T

In [ ]:
f1 = pd.read_csv(R / 'per_class_f1.csv')
f1[f1.test_crops > 0].sort_values('f1').head(20)  # weakest signs

## Real photographs (not scored: there are no labelled real seals)

In [ ]:
for f in sorted((R / 'figures').glob('*.jpg')):
    display(Markdown(f'**{f.stem}**'), Image(str(f), width=700))